# Experiment 5.3 — Synaptic history contextualization

Analysis-only notebook for the finalized 55-run sweep. Selection of the best single-tau, multi-tau, and rewriting conditions uses validation BA only; test metrics are displayed after selection. The main mechanistic tests are Hidden WholeCount accessibility, state reset, temporal shuffle, and offline relative-phase probes.


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts").is_dir() and (candidate / "notebooks").is_dir(): return candidate
    raise FileNotFoundError("Could not locate writingRing repository root")

ROOT = find_repo_root()
OUT = ROOT / "notebooks/artifacts/experiment_5_3_synaptic_contextual_evidence/frozen_local_synaptic_history_context_v1"
manifest = json.loads((OUT / "manifest.json").read_text(encoding="utf-8"))
runs = pd.read_csv(OUT / "runs.csv")
histories = pd.read_csv(OUT / "histories.csv")
probes = pd.read_csv(OUT / "probe_runs.csv")
phase_probes = pd.read_csv(OUT / "phase_probe_runs.csv")
ablations = pd.read_csv(OUT / "ablation_runs.csv")
local = pd.read_csv(OUT / "local_reference.csv")
assert manifest["expected_runs"] == 55 and len(runs) == 55
assert runs[["condition", "seed"]].drop_duplicates().shape[0] == 55
ORDER = ["direct","lif_tau242","lif_beta050","syn_single_s4","syn_single_s5","syn_single_s6","syn_multi_s23456","syn_multi_s456","syn_multi_s56","rsnn_beta050","factorized_rsnn_beta050"]
print(manifest["protocol_version"], OUT)


## Native accumulator and validation-only model selection

The LIF `tau_mem≈242 ms (beta=0.9375)` versus `beta=0.50` comparison is fixed a priori. Best single and multi conditions are selected by mean validation native accumulator BA only.


In [ ]:
summary = runs.groupby("condition").agg(n=("seed","nunique"),val_ba_mean=("native_val_balanced_accuracy","mean"),val_ba_sd=("native_val_balanced_accuracy","std"),test_ba_mean=("native_test_balanced_accuracy","mean"),test_ba_sd=("native_test_balanced_accuracy","std"),test_macro_f1_mean=("native_test_macro_f1","mean"),params=("trainable_parameters","first")).reindex(ORDER).reset_index()
SINGLE = ["syn_single_s4","syn_single_s5","syn_single_s6"]
MULTI = ["syn_multi_s23456","syn_multi_s456","syn_multi_s56"]
REWRITE = ["lif_tau242","lif_beta050",*SINGLE,*MULTI,"rsnn_beta050"]
def pick(candidates):
    return str(summary[summary.condition.isin(candidates)].sort_values(["val_ba_mean","condition"],ascending=[False,True]).iloc[0].condition)
selected_single_condition = pick(SINGLE)
selected_multi_condition = pick(MULTI)
selected_rewriting_condition = pick(REWRITE)
print("selected_single_condition:", selected_single_condition)
print("selected_multi_condition:", selected_multi_condition)
print("selected_rewriting_condition:", selected_rewriting_condition)
display(summary)
fig, ax = plt.subplots(figsize=(13,5.5)); x=np.arange(len(summary))
ax.bar(x,summary.test_ba_mean,yerr=summary.test_ba_sd,capsize=3); ax.set_xticks(x); ax.set_xticklabels(summary.condition,rotation=45,ha="right"); ax.set_ylabel("Test balanced accuracy"); ax.set_ylim(0,1); ax.set_title("Native signed-accumulator performance"); ax.grid(axis="y",alpha=.3); fig.tight_layout(); plt.show()


In [ ]:
def paired(left,right,metric="native_test_balanced_accuracy"):
    wide=runs[runs.condition.isin([left,right])].pivot(index="seed",columns="condition",values=metric).dropna()
    return pd.DataFrame({"seed":wide.index,"comparison":f"{left} - {right}","difference":wide[left]-wide[right]})
pairs=list(dict.fromkeys([("lif_tau242","lif_beta050"),("syn_single_s5","syn_single_s4"),("syn_single_s6","syn_single_s5"),("syn_multi_s56","syn_single_s5"),("syn_multi_s56","syn_single_s6"),("syn_multi_s456","syn_multi_s56"),("syn_multi_s23456","syn_multi_s456"),(selected_multi_condition,selected_single_condition),(selected_multi_condition,"rsnn_beta050"),("factorized_rsnn_beta050",selected_rewriting_condition)]))
paired_runs=pd.concat([paired(a,b) for a,b in pairs],ignore_index=True)
display(paired_runs.groupby("comparison").difference.agg(["mean","std","count"]).reset_index())


## Was history written into local evidence?

`context_writing_gain = HiddenWholeCount − LocalWholeCount`. The internalization ratio is a BA-based accessibility diagnostic, not mutual information. Relative10 is retained as a trajectory-information reference.


In [ ]:
pw=probes.pivot(index=["condition","seed"],columns="probe_type",values="test_ba").reset_index()
lw=local.pivot(index="seed",columns="probe_type",values="test_ba").reset_index()
mechanistic=pw.merge(lw,on="seed",validate="many_to_one")
mechanistic["context_writing_gain"]=mechanistic.hidden_whole_count-mechanistic.local_whole_count
mechanistic["local_timing_gap"]=mechanistic.local_relative10_ordered-mechanistic.local_whole_count
mechanistic["hidden_timing_gap"]=mechanistic.hidden_relative10_ordered-mechanistic.hidden_whole_count
mechanistic["internalization_ratio"]=mechanistic.context_writing_gain/mechanistic.local_timing_gap.replace(0,np.nan)
ms=mechanistic.groupby("condition").agg(hidden_whole_count=("hidden_whole_count","mean"),hidden_relative10=("hidden_relative10_ordered","mean"),context_writing_gain=("context_writing_gain","mean"),internalization_ratio=("internalization_ratio","mean")).reindex(ORDER).reset_index(); display(ms)
chosen=list(dict.fromkeys(["direct",selected_single_condition,selected_multi_condition,"rsnn_beta050","factorized_rsnn_beta050"]))
plot=mechanistic[mechanistic.condition.isin(chosen)]
fig,ax=plt.subplots(figsize=(11,5.5)); x=np.arange(len(chosen)); width=.25
for j,(col,label) in enumerate([("local_whole_count","Local WholeCount"),("hidden_whole_count","Hidden WholeCount"),("hidden_relative10_ordered","Hidden Relative10")]):
    values=plot.groupby("condition")[col].agg(["mean","std"]).reindex(chosen); ax.bar(x+(j-1)*width,values["mean"],width,yerr=values["std"],capsize=3,label=label)
ax.set_xticks(x); ax.set_xticklabels(chosen,rotation=35,ha="right"); ax.set_ylabel("Test BA"); ax.set_ylim(0,1); ax.set_title("Orderless history accessibility versus trajectory reference"); ax.legend(); ax.grid(axis="y",alpha=.3); fig.tight_layout(); plt.show()


## Causal ablations

Five temporal-shuffle replicates are averaged inside each `(condition, seed)` before cross-seed aggregation, so permutations do not inflate sample size. Both native accumulator BA and the newly fitted Hidden WholeCount probe are visualized.


In [ ]:
sh=ablations[ablations.ablation=="temporal_shuffle"].groupby(["condition","seed"]).agg(native_test_ba=("native_test_ba","mean"),hidden_whole_count_test_ba=("hidden_whole_count_test_ba","mean")).reset_index(); sh["ablation"]="temporal_shuffle"
ab=pd.concat([ablations[ablations.ablation!="temporal_shuffle"][["condition","seed","ablation","native_test_ba","hidden_whole_count_test_ba"]],sh],ignore_index=True)
asum=ab.groupby(["condition","ablation"]).agg(native_mean=("native_test_ba","mean"),native_sd=("native_test_ba","std"),probe_mean=("hidden_whole_count_test_ba","mean"),probe_sd=("hidden_whole_count_test_ba","std")).reset_index(); display(asum)
chosen=list(dict.fromkeys([selected_single_condition,selected_multi_condition,"rsnn_beta050","factorized_rsnn_beta050"])); kinds=["ordered","state_reset","temporal_shuffle"]
for mean_col,sd_col,ylabel,title in [("native_mean","native_sd","Native test BA","Causal dependence of the deployed accumulator"),("probe_mean","probe_sd","Hidden WholeCount test BA","Does order change contextualized evidence identity?")]:
    fig,ax=plt.subplots(figsize=(10.5,5.2)); x=np.arange(len(chosen)); width=.25
    for j,kind in enumerate(kinds):
        values=asum[(asum.ablation==kind)&asum.condition.isin(chosen)].set_index("condition").reindex(chosen); ax.bar(x+(j-1)*width,values[mean_col],width,yerr=values[sd_col],capsize=3,label=kind)
    ax.set_xticks(x); ax.set_xticklabels(chosen,rotation=30,ha="right"); ax.set_ylabel(ylabel); ax.set_ylim(0,1); ax.set_title(title); ax.legend(); ax.grid(axis="y",alpha=.3); fig.tight_layout(); plt.show()


## Relative-phase probes, activity, and validation learning curves

Each gesture contributes one mean representation per relative-progress bin. Phase labels are diagnostic only and never enter SNN training.


In [ ]:
phase_summary=phase_probes.groupby(["condition","probe_type"]).test_ba.mean().reset_index()
print("Frozen local phase probe:",local[local.probe_type=="phase_local"].test_ba.agg(["mean","std"]).to_dict())
chosen=list(dict.fromkeys([selected_single_condition,selected_multi_condition,"rsnn_beta050","factorized_rsnn_beta050"])); po=["phase_synaptic_state","phase_membrane","phase_spike","phase_contextual","phase_gate"]
pm=phase_summary[phase_summary.condition.isin(chosen)].pivot(index="condition",columns="probe_type",values="test_ba").reindex(index=chosen,columns=po)
fig,ax=plt.subplots(figsize=(10,4.5)); image=ax.imshow(pm.to_numpy(),aspect="auto"); ax.set_xticks(np.arange(len(po))); ax.set_xticklabels(po,rotation=35,ha="right"); ax.set_yticks(np.arange(len(chosen))); ax.set_yticklabels(chosen); ax.set_title("Relative-phase linear accessibility"); fig.colorbar(image,ax=ax,label="Test BA"); fig.tight_layout(); plt.show()
activity=runs.groupby("condition").agg(valid_events=("native_test_hidden_events_per_neuron_second","mean"),tail_fraction=("native_test_hidden_tail_event_fraction","mean"),tail_rate=("native_test_hidden_tail_events_per_neuron_second","mean"),evidence_scale=("native_test_evidence_abs_per_valid_step","mean")).reindex(ORDER).reset_index(); display(activity)
curves=["lif_tau242","lif_beta050",selected_single_condition,selected_multi_condition,"rsnn_beta050","factorized_rsnn_beta050"]
cs=histories[histories.condition.isin(curves)].groupby(["condition","epoch"]).val_balanced_accuracy.agg(["mean","std"]).reset_index()
fig,ax=plt.subplots(figsize=(11,6))
for condition in curves:
    c=cs[cs.condition==condition]; ax.plot(c.epoch,c["mean"],label=condition); ax.fill_between(c.epoch,c["mean"]-c["std"],c["mean"]+c["std"],alpha=.15)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation BA"); ax.set_ylim(0,1); ax.set_title("Validation learning curves"); ax.legend(ncol=2); ax.grid(alpha=.3); fig.tight_layout(); plt.show()


## Interpretation gate

A multi-tau result is treated as history contextualization only when Hidden WholeCount improves over the frozen local reference, the gain weakens under state reset and temporal shuffle, phase accessibility rises in an internal trajectory, and Hidden Relative10 does not collapse. This convergent pattern distinguishes history writing from a generic nonlinear remapping.
